# 加载模块

In [1]:
# 加载模块

from tqdm import tqdm
import akshare as ak
from vnpy.alpha.datafeed.akshare import StandardAkData
sd = StandardAkData()
# 获取沪深300成分股
from vnpy.trader.constant import Exchange, Interval
from vnpy.trader.object import HistoryRequest
from vnpy.alpha import AlphaLab, logger


In [2]:
# 设置下载参数
task_name = "HS300"
index_symbol = "000300"
index_vt_symbol = f"{index_symbol}.{sd.get_exchange(index_symbol)}"
start_date = "2020-01-01"
end_date = "2026-3-20"
lab = AlphaLab(f"./lab/{task_name}")

# 下载数据

In [ ]:
# 初始化数据服务（这里配置使用的RQData）
from vnpy.trader.datafeed import get_datafeed
datafeed = get_datafeed()
datafeed.init()
# 获取沪深300成分股
df = ak.index_stock_cons(symbol=index_symbol)
data = df["品种代码"].tolist()
vt_symbols = []
for symbol in data:
    exchange = sd.get_exchange(symbol)
    vt_symbols.append(f"{symbol}.{exchange}")
# 转换合约代码
index_components = {}
index_components[end_date] = vt_symbols
lab.save_component_data(index_symbol, index_components)

In [ ]:
# 除了成分股，还要下载指数数据
task_symbols = data + [index_symbol]

# 遍历下载数据
for vt_symbol in tqdm(task_symbols):
    symbol = vt_symbol

    req = HistoryRequest(symbol, Exchange("MIANA"), start_date, end_date, Interval.DAILY)
    req = sd.format_req(req=req)
    bars = datafeed.query_bar_history(req)

    if bars:
        lab.save_bar_data(bars)
    else:
        logger.error(f"下载{vt_symbol}数据失败")

# 策略模型设置

In [3]:
# 加载指数成分股代码
component_symbols = lab.load_component_symbols(index_symbol, start_date, end_date)


In [4]:
for vt_symbol in component_symbols:
    lab.add_contract_setting(
        vt_symbol,
        long_rate=5/10000,
        short_rate=5/10000,
        size=1,
        pricetick=0.0001,
    )


In [5]:
from vnpy.alpha.strategy import BacktestingEngine
from vnpy.alpha.strategy.strategies.ma_strategy import MA_Strategy
import polars as pl

engine = BacktestingEngine(lab)

strategy_setting = {}
# 添加回测参数配置
# noinspection PyTypeChecker
engine.set_parameters(
        vt_symbols=component_symbols,
        interval=Interval.DAILY,
        start=start_date,
        end=end_date,
        capital=1_000_000,
        risk_free=0.02,
        annual_days=240
    )

ma_setting = {"buy_ma":120,"sell_ma":20,"min_hold_days":1}
engine.add_strategy(MA_Strategy, ma_setting, pl.DataFrame())
 # 添加MA策略

# 回测

In [6]:
# 执行回测任务
engine.load_data()
engine.run_backtesting()
engine.calculate_result()
engine.calculate_statistics()
engine.show_chart()
engine.plot_all_stocks(selected_symbols=['300628.SZSE'])



2026-03-23 01:19:43.034 | INFO | Logger | 开始加载历史数据


100%|██████████| 280/280 [00:00<00:00, 286.30it/s]

2026-03-23 01:19:44.034 | INFO | Logger | 所有历史数据加载完成
2026-03-23 01:19:44.035 | INFO | Logger | 策略初始化完成
2026-03-23 01:19:44.035 | INFO | Logger | 开始回放历史数据


2026-03-23 01:19:50.230 | INFO | Logger | 历史数据回放结束
2026-03-23 01:19:50.231 | INFO | Logger | 开始计算逐日盯市盈亏
2026-03-23 01:19:50.383 | INFO | Logger | 逐日盯市盈亏计算完成
2026-03-23 01:19:50.383 | INFO | Logger | 开始计算策略统计指标
2026-03-23 01:19:50.384 | INFO | Logger | ------------------------------
2026-03-23 01:19:50.385 | INFO | Logger | 首个交易日：  2020-01-02
2026-03-23 01:19:50.385 | INFO | Logger | 最后交易日：  2026-03-20
2026-03-23 01:19:50.385 | INFO | Logger | 总交易日：  1504
2026-03-23 01:19:50.385 | INFO | Logger | 盈利交易日：  663
2026-03-23 01:19:50.385 | INFO | Logger | 亏损交易日：  721
2026-03-23 01:19:50.386 | INFO | Logger | 起始资金：  1,000,000.00
2026-03-23 01:19:50.386 | INFO | Logger | 结束资金：  830,564.95
2026-03-23 01:19:50.386 | INFO | Logger | 总收益率：  -16.94%
2026-03-23 01:19:50.386 | INFO | Logger | 年化收益：  -2.70%
2026-03-23 01:19:50.386 | INFO | Logger | 最大回撤:   -754,381.29
2026-03-23 01:19:50.387 | INFO | Logger | 百分比最大回撤: -55.43%
2026-03-23 01:19:50.387 | INFO | Logger | 最长回撤天数:   1517
2026-03-23 01:19:50.

2026-03-23 01:19:50.462 | INFO | Logger | 正在绘制 1 只股票的价格走势...
2026-03-23 01:19:50.471 | INFO | Logger | 正在绘制 300628.SZSE 的价格走势，共 1504 个数据点
